# 01. ACOS Setup, Pretrained Model Caching & Exploratory Data Analysis (EDA)

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook handles the initial setup for running ACOS on **Google Colab** (or Local environment), checks GPU acceleration, downloads and caches the pretrained `bert-base-uncased` model assets locally (to eliminate legacy S3 download failures), and performs comprehensive Exploratory Data Analysis (EDA) with publication-quality visualizations and CSV statistical exports.

## 1. Environment Setup & Dependency Installation
Install required dependencies (`torchcrf`, `transformers`, `huggingface_hub`, `seaborn`, `scikit-learn`).

In [ ]:
# Check if running on Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🚀 Running in Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("💻 Running in Local environment.")

# Install dependencies
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3

import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU Availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected. Running on CPU.")

## 2. Directory Navigation & Path Initialization

In [ ]:
# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 3. Import colab_utils with fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 3. Initialize Timestamped Session Directory (`DDMMYYYY_HMS`)
Every session automatically creates an isolated directory inside `results/` for storing plots, CSV tables, model checkpoints, and execution logs.

In [ ]:
# Choose domain: 'rest16' (Restaurant-ACOS) or 'laptop' (Laptop-ACOS)
DOMAIN = "rest16"

results_base = os.path.join(base_project_dir, "results")
session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)

print("Directory structure:")
for k, v in session_dirs.items():
    print(f"  - {k}: {v}")
# Direktori Markdown untuk hasil teks tiap step
md_dir = session_dirs["md"]
plots_dir = session_dirs["plots"]
csv_dir = session_dirs["csv"]
logs_dir = session_dirs["logs"]

# Akumulator laporan Markdown untuk notebook ini
rep = MarkdownReport(
    f"01 - Setup & EDA [{DOMAIN.upper()}]",
    md_dir,
    filename="01_setup_dan_eda.md",
    meta={"domain": DOMAIN, "session_dir": session_dirs["root"], "device": str(device)},
)
print(f"[md] Hasil teks notebook ini akan ditulis ke: {md_dir}")


## 4. Download & Cache Pretrained BERT Model (`bert-base-uncased`)
Downloads `config.json`, `pytorch_model.bin`, and `vocab.txt` directly from HuggingFace Hub to local cache `./bert_base_uncased`.

In [ ]:
bert_cache_dir = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_cache_dir)

# Verify files
for f in ["config.json", "pytorch_model.bin", "vocab.txt"]:
    fpath = os.path.join(bert_cache_dir, f)
    assert os.path.exists(fpath), f"Missing BERT file: {fpath}"
    print(f"✅ {f} ({os.path.getsize(fpath)/1024/1024:.2f} MB)")

## 5. Exploratory Data Analysis (EDA) & Visualization
Perform deep statistical analysis on `Restaurant-ACOS` (`rest16`) and `Laptop-ACOS` (`laptop`) datasets:
- Sentence and Quadruple counts per split (Train / Dev / Test)
- Explicit vs. Implicit Aspect and Opinion proportions
- Top Aspect Categories & Sentiment Polarity distributions
- Automatic export to high-resolution PNG plots (`plots/`) and CSV files (`csv/`).

In [ ]:
data_root = os.path.join(base_project_dir, "data")

df_stats, df_records = analyze_and_plot_eda(
    data_dir=data_root,
    domain=DOMAIN,
    output_plots_dir=plots_dir,
    output_csv_dir=csv_dir,
)

rep.section("1. Statistik dataset per split")

if df_stats is None or df_stats.empty:
    print("[peringatan] Statistik EDA kosong: cek keberadaan folder data/.")
    rep.text("Statistik EDA kosong. Folder `data/` tidak ditemukan atau kosong.")
else:
    export_step_table(
        df_stats,
        name="eda_01_statistik_per_split",
        csv_dir=csv_dir,
        md_dir=md_dir,
        title=f"Statistik Dataset {DOMAIN.upper()} per Split",
        notes="Kolom sentimen: 0 = negative, 1 = neutral, 2 = positive.",
    )
    rep.table(df_stats, caption="Statistik dasar per split")


### Sample Data Preview with Implicit/Explicit Flags

In [ ]:
if df_records is None or df_records.empty:
    print("[peringatan] Tidak ada record quadruple untuk dianalisis.")
else:
    print(f"Total record quadruple: {len(df_records):,}")

    # 1. Preview sampel beranotasi
    export_step_table(
        df_records.head(25),
        name="eda_02_preview_sampel",
        csv_dir=csv_dir,
        md_dir=md_dir,
        title=f"Preview 25 Quadruple Beranotasi ({DOMAIN.upper()})",
        max_rows_md=25,
    )

    # 2. Rekap implicit vs explicit
    n = len(df_records)
    imp_rows = []
    for label, col in [("Implicit Aspect", "Is_Implicit_Aspect"),
                       ("Implicit Opinion", "Is_Implicit_Opinion")]:
        cnt = int(df_records[col].sum())
        imp_rows.append({"Tipe": label, "Jumlah": cnt,
                         "Persen": round(cnt / n * 100, 2),
                         "Total_Quadruple": n})
    both = int((df_records["Is_Implicit_Aspect"] & df_records["Is_Implicit_Opinion"]).sum())
    imp_rows.append({"Tipe": "Implicit Aspect + Implicit Opinion", "Jumlah": both,
                     "Persen": round(both / n * 100, 2), "Total_Quadruple": n})
    df_imp = pd.DataFrame(imp_rows)

    rep.section("2. Proporsi implicit vs explicit")
    export_step_table(
        df_imp, name="eda_03_rekap_implicit", csv_dir=csv_dir, md_dir=md_dir,
        title=f"Rekap Implicit Aspect/Opinion ({DOMAIN.upper()})",
    )
    rep.table(df_imp, caption="Proporsi implicit")

    # 3. Distribusi kategori lengkap
    df_cat = (df_records["Category"].value_counts()
              .rename_axis("Category").reset_index(name="Jumlah"))
    df_cat["Persen"] = (df_cat["Jumlah"] / n * 100).round(2)

    rep.section("3. Distribusi kategori aspek")
    export_step_table(
        df_cat, name="eda_04_distribusi_kategori", csv_dir=csv_dir, md_dir=md_dir,
        title=f"Distribusi Kategori Aspek ({DOMAIN.upper()}) - {len(df_cat)} kategori",
        max_rows_md=25,
    )
    rep.table(df_cat.head(15), caption="15 kategori terbanyak")

    # 4. Distribusi sentimen
    senti_names = {0: "negative (0)", 1: "neutral (1)", 2: "positive (2)"}
    df_senti = (df_records["Sentiment"].map(senti_names).value_counts()
                .rename_axis("Sentimen").reset_index(name="Jumlah"))
    df_senti["Persen"] = (df_senti["Jumlah"] / n * 100).round(2)

    rep.section("4. Distribusi polaritas sentimen")
    export_step_table(
        df_senti, name="eda_05_distribusi_sentimen", csv_dir=csv_dir, md_dir=md_dir,
        title=f"Distribusi Sentimen ({DOMAIN.upper()})",
    )
    rep.table(df_senti, caption="Polaritas sentimen")

    # 5. Statistik panjang kalimat
    df_len = df_records["Text_Length"].describe().to_frame("Nilai").reset_index()
    df_len.columns = ["Statistik", "Nilai"]

    rep.section("5. Statistik panjang kalimat")
    export_step_table(
        df_len, name="eda_06_statistik_panjang", csv_dir=csv_dir, md_dir=md_dir,
        title=f"Statistik Panjang Kalimat dalam Kata ({DOMAIN.upper()})",
    )
    rep.table(df_len, caption="Panjang kalimat (kata)")


### Display Generated Visualization Charts
Visualizations are rendered inline and saved to `plots/`.

In [ ]:
from IPython.display import Image, display

eda_plots = [
    ("01_eda_dataset_distribution.png", "Komposisi dataset & explicit vs implicit aspect"),
    ("02_eda_category_sentiment.png", "Kategori teratas & breakdown polaritas sentimen"),
    ("02b_eda_length_and_implicit_combo.png", "Distribusi panjang kalimat & kombinasi implicit/explicit"),
    ("02c_eda_category_sentiment_heatmap.png", "Heatmap kategori (top 12) x sentimen"),
]

rep.section("6. Visualisasi")
shown = 0
for fname, caption in eda_plots:
    path = os.path.join(plots_dir, fname)
    if os.path.exists(path):
        print(f"[plot] {caption}")
        display(Image(path))
        rep.image(path, caption)
        shown += 1
    else:
        print(f"[plot] Tidak ditemukan (dilewati): {fname}")

print(f"
{shown}/{len(eda_plots)} plot ditampilkan.")


## 6. Summary of Exported Artifacts
All outputs from this step are safely persisted in the timestamped session directory:

In [ ]:
def _list_dir(label, path):
    rows = []
    if os.path.isdir(path):
        for f in sorted(os.listdir(path)):
            fp = os.path.join(path, f)
            if os.path.isfile(fp):
                rows.append({"Jenis": label, "Nama": f,
                             "Ukuran_KB": round(os.path.getsize(fp) / 1024, 1)})
    return rows

artefak = (_list_dir("CSV", csv_dir) + _list_dir("Plot", plots_dir)
           + _list_dir("Markdown", md_dir) + _list_dir("Log", logs_dir))
df_art = pd.DataFrame(artefak)

rep.section("7. Artefak yang dihasilkan")
if not df_art.empty:
    export_step_table(
        df_art, name="eda_07_daftar_artefak", csv_dir=csv_dir, md_dir=md_dir,
        title="Daftar Artefak Sesi EDA", max_rows_md=100,
    )
    rep.table(df_art, max_rows=100, caption="Artefak sesi")

rep.text(f"Sesi: `{session_dirs['root']}`")
report_path = rep.save()

print(f"
Laporan Markdown notebook 01: {report_path}")
print("Lanjut ke '02_ACOS_Step1_Aspect_Opinion_Extraction.ipynb'.")
